# **(4)** Chemeleon (molecular) pretraing

Using the [CheMeleon](https://github.com/JacksonBurns/chemeleon) pretrained model to predict peptides properties (labels: `purity`, `camsol_score`, `revised_50hemo`).

```
chemeleon_mp.pt  (large pretrained model on molecular descriptors - optimised for small molecules properties predictions)
        │
        ├─ BondMessagePassing   → LOADED  (Chemeleon weights)
        └─ RegressionFFN        → NEW     (3 heads)
                │
                ▼
        [purity,  camsol_score,  revised_50hemo]
```

> **Reference:** [finetuning_demo.ipynb](https://github.com/JacksonBurns/chemeleon/blob/main/finetuning_demo.ipynb)


In [ ]:
pip install chemprop lightning rdkit pandas chemeleon

In [ ]:
import os

if os.getenv("COLAB_RELEASE_TAG"):
    if not os.path.exists("IX_technical_turorial2026"):
        !git clone https://github.com/rbirolo/IX_technical_turorial2026.git
    
    %cd IX_technical_turorial2026

In [ ]:
import warnings, pickle
warnings.filterwarnings("ignore")
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import lightning as pl
from scipy import stats as sp_stats
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")

from chemprop import data, models, featurizers, nn

## Download Chemeleon Backbone

The model is hosted on Zenodo.


In [ ]:
import requests
import time
from pathlib import Path

CHEMELEON_PT = Path("chemeleon_mp.pt")
url = "https://zenodo.org/records/15460715/files/chemeleon_mp.pt"

if not CHEMELEON_PT.exists():
    for attempt in range(3):
        try:
            print(f"Download attempt {attempt + 1}/3...")

            r = requests.get(url, stream=True, timeout=60)
            r.raise_for_status()

            with open(CHEMELEON_PT, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)

            print(f"Downloaded successfully: {CHEMELEON_PT}")
            break

        except requests.RequestException as e:
            print(f"Download failed: {e}")

            if attempt < 4:
                time.sleep(5)
            else:
                raise
else:
    print(f"Found existing {CHEMELEON_PT}, skipping download.")


## Load Chemeleon Backbone

Chemeleon's `.pt` stores `hyper_parameters` and `state_dict` separately.  
We rebuild `BondMessagePassing` from the saved hyper-parameters so the  
architecture matches exactly.


In [ ]:
chemeleon_ckpt = torch.load(CHEMELEON_PT, weights_only=False)

print("Chemeleon hyper_parameters:")
for k, v in chemeleon_ckpt["hyper_parameters"].items():
    print(f"  {k:<20} {v}")

# Build MP from saved hyper-parameters + load weights
mp  = nn.BondMessagePassing(**chemeleon_ckpt["hyper_parameters"])
mp.load_state_dict(chemeleon_ckpt["state_dict"])

agg        = nn.MeanAggregation()
featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()

print(f"\nBackbone output dim : {mp.output_dim}")
print(f"Backbone params     : {sum(p.numel() for p in mp.parameters()):,}")

## Configuration

In [ ]:
FT_data_file = "data/experimental_dataset.csv"
FT_output    = Path("chemeleon_ft_model")
FT_targets = ["purity", "camsol_score", "revised_50hemo"]

FFN_LAYERS = 2
FFN_DIM    = 300
DROPOUT    = 0.2

MAX_EPOCHS = 50
BATCH_SIZE = 64
INIT_LR    = 1e-4
MAX_LR     = 1e-3
FINAL_LR   = 1e-4
WARMUP_EPS = 2
SEED       = 42
SPLIT_SIZES = (0.70, 0.15, 0.15)

## Load Data

Note: `MoleculeDatapoint.from_smi()` is used (Chemeleon) instead of  
passing a pre-built `mol` object.


In [ ]:
def load_dataset(path, targets):
    df = pd.read_csv(path)
    for t in targets:
        df[t] = pd.to_numeric(df[t], errors="coerce")
    df = df.dropna(subset=["SMILES"]).copy()

    datapoints, valid_idx = [], []
    for i, row in df.iterrows():
        mol = Chem.MolFromSmiles(str(row["SMILES"]))
        if mol is None:
            print(f"Invalid SMILES (row {i}): {str(row['SMILES'])[:80]}…")
            continue
        y = row[targets].values.astype(float)
        datapoints.append(data.MoleculeDatapoint.from_smi(str(row["SMILES"]), y))
        valid_idx.append(i)

    df = df.iloc[valid_idx].reset_index(drop=True)
    ys = df[targets].values.astype(float)
    return datapoints, [dp.mol for dp in datapoints], ys, df

all_datapoints, mols, ys, df_clean = load_dataset(FT_data_file, FT_targets)
N = len(all_datapoints)

print(f"{'Target':<22} {'non-NaN':>8} {'NaN':>6} {'mean':>9} {'std':>8}")
print("─" * 58)
for i, t in enumerate(FT_targets):
    v = ys[:, i][~np.isnan(ys[:, i])]
    print(f"  {t:<20} {len(v):>8} {N-len(v):>6} {v.mean():>9.3f} {v.std():>8.3f}")

## Split and DataLoaders

In [ ]:
pl.seed_everything(SEED)

train_idxs, val_idxs, test_idxs = data.make_split_indices(
    mols, "random", SPLIT_SIZES, seed=SEED
)

train_data, val_data, test_data = data.split_data_by_indices(
    all_datapoints, train_idxs, val_idxs, test_idxs
)

train_ds = data.MoleculeDataset(train_data[0], featurizer)
val_ds   = data.MoleculeDataset(val_data[0],   featurizer)
test_ds  = data.MoleculeDataset(test_data[0],  featurizer)

scaler = train_ds.normalize_targets()
val_ds.normalize_targets(scaler)

train_loader = data.build_dataloader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = data.build_dataloader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = data.build_dataloader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

_ti = train_idxs[0]; _vi = val_idxs[0]; _tei = test_idxs[0]
print(f"Train : {len(_ti):>5}  ({len(_ti)/N*100:.1f}%)")
print(f"Val   : {len(_vi):>5}  ({len(_vi)/N*100:.1f}%)")
print(f"Test  : {len(_tei):>5}  ({len(_tei)/N*100:.1f}%)")

## Build Model


In [ ]:
ffn = nn.RegressionFFN(
    n_tasks          = len(FT_targets),
    input_dim        = mp.output_dim,   # determined by Chemeleon
    hidden_dim       = FFN_DIM,
    n_layers         = FFN_LAYERS,
    dropout          = DROPOUT,
)

mpnn = models.MPNN(
    mp, agg, ffn,
    batch_norm = False,
    metrics    = [nn.metrics.RMSE(), nn.metrics.MAE()],
    init_lr    = INIT_LR,
    max_lr     = MAX_LR,
    final_lr   = FINAL_LR,
    warmup_epochs = WARMUP_EPS,
)

total     = sum(p.numel() for p in mpnn.parameters())
trainable = sum(p.numel() for p in mpnn.parameters() if p.requires_grad)
print(f"Total params     : {total:,}")
print(f"Trainable params : {trainable:,}")
print(f"\nFFN input_dim = mp.output_dim = {mp.output_dim}")

## Freeze Chemeleon Backbone

For small dataset to preserve the foundation representations, freeze the backbone and train only the FFN.  
**Comment this cell out** to fine-tune the full model end-to-end.


In [ ]:
for name, param in mpnn.named_parameters():
    if name.startswith("message_passing"):
        param.requires_grad = False

trainable = sum(p.numel() for p in mpnn.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in mpnn.parameters() if not p.requires_grad)
print(f"Trainable : {trainable:,} params  (FFN only)")
print(f"Frozen    : {frozen:,} params  (Chemeleon backbone)")

## Training

In [ ]:
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

early_stop = EarlyStopping(
    monitor   = "val_loss",
    patience  = 10,
    mode      = "min",
    verbose   = True,
)

checkpoint = ModelCheckpoint(
    monitor="val_loss",
    mode="min", save_top_k=1,
    filename="best_model",
    )

trainer = pl.Trainer(
    max_epochs           = MAX_EPOCHS,
    logger               = False,
    enable_progress_bar  = True,
    accelerator          = "auto",
    callbacks            = [early_stop, checkpoint],
)

trainer.fit(mpnn, train_loader, val_loader)
stopped_at = trainer.current_epoch

print(f"\nStopped at epoch {trainer.current_epoch} / {MAX_EPOCHS}")
print(f"Best val_loss: {checkpoint.best_model_score:.5f}")
best_ckpt = torch.load( checkpoint.best_model_path, weights_only=False, )
mpnn.load_state_dict(best_ckpt["state_dict"])

## Save

In [ ]:
FT_output.mkdir(parents=True, exist_ok=True)

torch.save(mpnn.state_dict(), FT_output / "chemeleon_ft_earlystopping.pt")

with open(FT_output / "chemeleon_ft_earlystopping_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

## Evaluation on test set


In [ ]:
def evaluate(model, loader, target_names):
    model.eval()
    all_p, all_gt = [], []
    with torch.no_grad():
        for batch in loader:
            bmg, V_d, X_d, targets, *_ = batch
            all_p.append(model(bmg, V_d, X_d).numpy())
            all_gt.append(targets.numpy())

    preds   = scaler.inverse_transform(np.concatenate(all_p))
    gts = np.concatenate(all_gt)

    out = {}
    for i, t in enumerate(target_names):
        mask = ~np.isnan(gts[:, i])
        if mask.sum() < 2: continue
        gt, pr = gts[mask, i], preds[mask, i]
        out[t] = {
            "gt": gt, "pr": pr, "n": int(mask.sum()),
            "rmse"      : float(np.sqrt(mean_squared_error(gt, pr))),
            "mae"       : float(mean_absolute_error(gt, pr)),
            "pearson_r" : float(sp_stats.pearsonr(gt, pr)[0]),
            "spearman_r": float(pd.Series(gt).corr(pd.Series(pr), method="spearman")),
        }
    return out

metrics = evaluate(mpnn, test_loader, FT_targets)

print(f"{'Target':<22} {'RMSE':>8} {'MAE':>8} {'Pearson':>9} {'Spearman':>10} {'N':>6}")
print("─" * 76)
for t in FT_targets:
    if t not in metrics:
        print(f"  {t:<20}  — no data"); continue
    m = metrics[t]
    print(f"  {t:<20}  {m['rmse']:>8.4f} {m['mae']:>8.4f}"
          f" {m['pearson_r']:>9.4f} {m['spearman_r']:>10.4f} {m['n']:>6}")

In [ ]:
colors = ["#2196F3", "#4CAF50", "#E91E63"]
fig, axes = plt.subplots(1, len(FT_targets), figsize=(5.5 * len(FT_targets), 5))

for ax, t, c in zip(axes, FT_targets, colors):
    if t not in metrics:
        ax.set_visible(False); continue
    m  = metrics[t]
    lo = min(m["gt"].min(), m["pr"].min()) * 0.95
    hi = max(m["gt"].max(), m["pr"].max()) * 1.05
    ax.plot([lo, hi], [lo, hi], "k--", lw=1, zorder=1)
    ax.scatter(m["gt"], m["pr"], color=c, edgecolors="white", s=60,
               linewidths=0.5, alpha=0.85, zorder=2,
               label=f"RMSE={m['rmse']:.3f}\nPearson r ={m['pearson_r']:.3f}\nN={m['n']}")
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_title(t, fontsize=12, fontweight="normal")
    ax.set_xlabel("y_value"); ax.set_ylabel("predicted")
    ax.legend(fontsize=8); ax.spines[["top","right"]].set_visible(False)

plt.tight_layout(); plt.show()